# Phase 2 - Exploratory Data Analysis
This notebook covers Ratings Analysis, User Behavior, Movie Popularity, Genre Analysis, Temporal Trends, and Sparsity Calculation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visual aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Ensure outputs directory exists
os.makedirs('../outputs/figures', exist_ok=True)

# Load data
print("Loading datasets...")
try:
    ratings = pd.read_csv('../data/raw/raw_ratings.csv')
    movies = pd.read_csv('../data/raw/raw_movies.csv')
    print(f"Loaded {len(ratings):,} ratings and {len(movies):,} movies.")
except FileNotFoundError:
    print("Data files not found. Please ensure Phase 1 data collection or mock generation has been run.")

## 1. Ratings Analysis
Plot the distribution of all ratings (1.0 to 5.0 in 0.5 steps). Find the mean, median, and standard deviation. Check if ratings are skewed toward higher values.

In [ ]:
mean_rating = ratings['rating'].mean()
median_rating = ratings['rating'].median()
std_rating = ratings['rating'].std()

print(f"Mean Rating: {mean_rating:.3f}")
print(f"Median Rating: {median_rating:.3f}")
print(f"Standard Deviation: {std_rating:.3f}")

plt.figure()
sns.countplot(data=ratings, x='rating', color='skyblue')
plt.title('Distribution of Movie Ratings')
plt.xlabel('Rating (0.5 to 5.0)')
plt.ylabel('Count')
plt.savefig('../outputs/figures/eda_ratings_distribution.png', bbox_inches='tight')
plt.show()

## 2. User Behavior
Plot a histogram of how many ratings each user has given. Identify the expected long tail.

In [ ]:
user_ratings_count = ratings.groupby('userId').size()

plt.figure()
sns.histplot(user_ratings_count, bins=100, color='coral', log_scale=(False, True))
plt.title('Histogram of Ratings per User (Log Scale Y)')
plt.xlabel('Number of Ratings Given by a User')
plt.ylabel('Count of Users (Log Scale)')
plt.savefig('../outputs/figures/eda_user_behavior.png', bbox_inches='tight')
plt.show()

## 3. Movie Popularity
Plot a histogram of how many ratings each movie has received to observe the cold-start problem.

In [ ]:
movie_ratings_count = ratings.groupby('movieId').size()

plt.figure()
sns.histplot(movie_ratings_count, bins=100, color='mediumseagreen', log_scale=(False, True))
plt.title('Histogram of Ratings per Movie (Log Scale Y)')
plt.xlabel('Number of Ratings a Movie Received')
plt.ylabel('Count of Movies (Log Scale)')
plt.savefig('../outputs/figures/eda_movie_popularity.png', bbox_inches='tight')
plt.show()

## 4. Genre Analysis
Parse the pipe-separated genre column and plot genre frequency.

In [ ]:
all_genres = movies['genres'].str.split('|').explode()
genre_counts = all_genres.value_counts()

plt.figure(figsize=(10, 8))
sns.barplot(x=genre_counts.values, y=genre_counts.index, hue=genre_counts.index, legend=False, palette='viridis')
plt.title('Frequency of Genres in Dataset')
plt.xlabel('Number of Movies')
plt.ylabel('Genre')
plt.savefig('../outputs/figures/eda_genre_frequency.png', bbox_inches='tight')
plt.show()

## 5. Temporal Trends
Convert timestamps to datetime and plot average rating per year.

In [ ]:
ratings['datetime'] = pd.to_datetime(ratings['timestamp'], unit='s')
ratings['year'] = ratings['datetime'].dt.year

yearly_avg_rating = ratings.groupby('year')['rating'].mean()

plt.figure()
sns.lineplot(x=yearly_avg_rating.index, y=yearly_avg_rating.values, marker='o', color='purple')
plt.title('Average Rating per Year')
plt.xlabel('Year')
plt.ylabel('Average Rating')
plt.grid(True)
plt.savefig('../outputs/figures/eda_temporal_trends.png', bbox_inches='tight')
plt.show()

## 6. Sparsity Calculation
Calculate matrix sparsity.

In [ ]:
total_ratings = len(ratings)
total_users = ratings['userId'].nunique()
total_movies = ratings['movieId'].nunique()

sparsity = 1.0 - (total_ratings / (total_users * total_movies))

print("--- Sparsity Calculation ---")
print(f"Total Ratings: {total_ratings:,}")
print(f"Total Users: {total_users:,}")
print(f"Total Movies Rated: {total_movies:,}")
print(f">>> Sparsity: {sparsity:.5%} <<<")
